In [345]:
import torch
import torch.nn as nn
from torch_geometric.nn import GCNConv,GATConv
import pandas as pd
import itertools
from collections import defaultdict
import tqdm as tqdm

In [346]:
video_concept_data = pd.read_csv("./data/video_concept.csv")
user_video_data = pd.read_csv("./data/user_video.csv")
course_video_data = pd.read_csv("./data/course_video.csv")

In [347]:
all_course_id = sorted(course_video_data['course_id'].unique())
all_video_id = sorted(video_concept_data['video_id'].unique())
all_concept_id = sorted(video_concept_data['concept_id'].unique())

course_map = {ci: i for i, ci in enumerate(all_course_id)}
video_map = {vi: i for i, vi in enumerate(all_video_id)}
concept_map = {ci: i for i, ci in enumerate(all_concept_id)}

num_course = len(all_course_id)
num_video = len(all_video_id)
num_concept = len(all_concept_id)

# Course relational Graph

In [348]:
#构建course-relational graph
course_rela_map = defaultdict(list)
for _, row in course_video_data.iterrows():
    course_rela_map[course_map[row['course_id']]].append(video_map[row['video_id']])

In [324]:
# 2. 为同一课程下的所有视频对创建边
course_edges = []
for _, videos in course_rela_map.items():
    for v1, v2 in itertools.combinations(videos, 2):
        course_edges.append([v1, v2])
        course_edges.append([v2, v1]) # 添加反向边，构成无向图

# 转换为 PyG 需要的 edge_index 格式
edge_index_course = torch.tensor(course_edges, dtype=torch.long).t().contiguous()
print(f"课程关系图构建完毕，边索引形状: {edge_index_course.shape}")

课程关系图构建完毕，边索引形状: torch.Size([2, 339612])


In [325]:
print(edge_index_course[:,:50])

tensor([[ 0,  1,  0,  2,  0,  3,  0,  4,  0,  5,  0,  6,  0,  7,  0,  8,  0,  9,
          0, 10,  0, 11,  0, 12,  0, 13,  0, 14,  0, 15,  0, 16,  0, 17,  0, 18,
          0, 19,  0, 20,  0, 21,  0, 22,  0, 23,  0, 24,  0, 25],
        [ 1,  0,  2,  0,  3,  0,  4,  0,  5,  0,  6,  0,  7,  0,  8,  0,  9,  0,
         10,  0, 11,  0, 12,  0, 13,  0, 14,  0, 15,  0, 16,  0, 17,  0, 18,  0,
         19,  0, 20,  0, 21,  0, 22,  0, 23,  0, 24,  0, 25,  0]])


# Konwledge concept relational Graph

In [326]:
#构建knowledge concept-relational graph
konwledge_concept_rela_map = defaultdict(set)
for _, row in video_concept_data.iterrows():
    konwledge_concept_rela_map[video_map[row['video_id']]].add(concept_map[row['concept_id']])

In [327]:
print(konwledge_concept_rela_map)

defaultdict(<class 'set'>, {0: {0, 1, 2, 3, 4, 5, 6, 7, 8, 9}, 1: {5, 8, 10, 11, 12, 13, 14, 15, 16, 17}, 2: {8, 10, 11, 16, 18, 19, 20, 21, 22, 23}, 3: {8, 12, 24, 25, 26, 27, 28, 29, 30, 31}, 4: {32, 33, 34, 35, 36, 37, 38, 39, 8, 23}, 5: {40, 41, 42, 43, 8, 44, 15, 16, 21, 31}, 6: {3, 5, 6, 8, 12, 45, 46, 47, 48, 49}, 7: {1, 40, 8, 11, 12, 50, 51, 52, 53, 54}, 8: {1, 34, 33, 4, 35, 8, 22, 55, 56, 57}, 9: {40, 8, 63, 22, 58, 59, 60, 61, 62, 31}, 10: {64, 65, 66, 67, 68, 69, 40, 8, 11, 12}, 11: {70, 71, 72, 73, 74, 8, 12, 75, 50, 31}, 12: {5, 70, 8, 74, 75, 76, 77, 12, 27, 31}, 13: {1, 5, 7, 8, 9, 46, 78, 79, 80, 53}, 14: {8, 15, 81, 82, 83, 84, 85, 86, 87, 88}, 15: {67, 4, 5, 8, 12, 89, 90, 91, 92, 31}, 16: {69, 8, 11, 75, 13, 14, 15, 19, 93, 94}, 17: {32, 96, 97, 98, 69, 8, 11, 12, 47, 95}, 18: {99, 100, 101, 40, 8, 74, 90, 63, 62, 31}, 19: {102, 103, 104, 8, 105, 11, 75, 45, 106, 58}, 20: {5, 8, 11, 107, 108, 109, 110, 111, 50, 54}, 21: {37, 40, 11, 14, 47, 112, 113, 114, 115, 22},

In [328]:
def jaccard_similarity(set1, set2):
    """
    J(A, B) = |A ∩ B| / |A ∪ B|
    """
    intersection = len(set1.intersection(set2))
    union = len(set1.union(set2))
    return intersection / union if union > 0 else 0

video_ids = list(konwledge_concept_rela_map.keys())
similarity_threshold = 0.2  # 相似度阈值


In [329]:
concept_edges = []
for c1, c2 in itertools.combinations(video_ids, 2):
    concepts1 = konwledge_concept_rela_map[c1]
    concepts2 = konwledge_concept_rela_map[c2]
    sim = jaccard_similarity(concepts1, concepts2)
    
    if sim > similarity_threshold:
        concept_edges.append([c1, c2])
        concept_edges.append([c2, c1])

edge_index_concept = torch.tensor(concept_edges, dtype=torch.long).t().contiguous()


In [330]:
print(edge_index_concept.size())
print(edge_index_concept[:,:50])

torch.Size([2, 18270])
tensor([[   0,    6,    0,   13,    0,   25,    0,   28,    0,   30,    0, 1050,
            0, 1078,    1,    2,    1,   16,    1,   27,    1,   56,    1,   66,
            1,  227,    3,   12,    3,   26,    3,   48,    3,   60,    3,   63,
            4,    8,    4,   47,    5,   29,    5,   51,    5,  155,    5,  204,
            5, 1076],
        [   6,    0,   13,    0,   25,    0,   28,    0,   30,    0, 1050,    0,
         1078,    0,    2,    1,   16,    1,   27,    1,   56,    1,   66,    1,
          227,    1,   12,    3,   26,    3,   48,    3,   60,    3,   63,    3,
            8,    4,   47,    4,   29,    5,   51,    5,  155,    5,  204,    5,
         1076,    5]])


# Video relational Graph

In [331]:
#构建video_relational graph
#按照用户ID和开始时间进行排序
video_start_time = pd.to_datetime(user_video_data['start_time'])
video_sorted = user_video_data.sort_values(by=['user_id','start_time'])

In [332]:
#每个用户所观看的视频
#{user_id:[1,2,3,...]}
video_rela_map = defaultdict(list)
for _, row in video_sorted.iterrows():
    video_rela_map[row['user_id']].append(video_map[row['video_id']])

In [333]:
print([[user_id, seq[:5]] for user_id, seq in list(video_rela_map.items())[:2] ])

[[1, [938, 987, 983, 960, 973]], [2, [1629, 1630, 1681, 1313, 1312]]]


In [334]:
def video_relational_graph(user_id):
    """
    为一个特定用户动态构建视频关系图。
    """
    #每个用户所看的视频
    sequence = video_rela_map.get(user_id, [])
    video_edges = []
    #u,v 表示用户看完u后看了v
    video_edges = [[sequence[i], sequence[i+1]] for i in range(len(sequence) - 1)]    
    return torch.tensor(video_edges, dtype=torch.long).t().contiguous()

edge_index_user1 = video_relational_graph(1)

print("\n--- 动态视频关系图 (Gv) 构建示例 ---")
print(f"用户 1 的图:")
print(f"  边数: {edge_index_user1.shape[1]}")
print(f"  边索引 (前5条): {edge_index_user1[:, :5]}")



--- 动态视频关系图 (Gv) 构建示例 ---
用户 1 的图:
  边数: 99
  边索引 (前5条): tensor([[938, 987, 983, 960, 973],
        [987, 983, 960, 973, 962]])


In [335]:
class LSTM_layer (nn.Module):
    """
    严格遵循 TOME 论文公式 (7)-(11) 实现的定制 LSTM 单元。
    """
    def __init__(self, input_dim, context_dim, hidden_dim):
        super(LSTM_layer, self).__init__()
        # 这里的 input_dim 对应论文中的 e^r_t (上一时刻的输出)
        # context_dim 对应 e^C_t, e^K_t, e^V_t
        # hidden_dim 对应 h_{t-1}
        
        # --- 输入门 (Input Gate) - 公式 (7) ---
        self.W_i = nn.Linear(input_dim, hidden_dim, bias=True)
        self.U_i = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.V_ic = nn.Linear(context_dim, hidden_dim, bias=False)
        self.Z_ik = nn.Linear(context_dim, hidden_dim, bias=False)
        self.Q_iv = nn.Linear(context_dim, hidden_dim, bias=False)
        
        # --- 遗忘门 (Forget Gate) - 公式 (8) ---
        self.W_f = nn.Linear(input_dim, hidden_dim, bias=True)
        self.U_f = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.V_fc = nn.Linear(context_dim, hidden_dim, bias=False)
        self.Z_fk = nn.Linear(context_dim, hidden_dim, bias=False)
        self.Q_fv = nn.Linear(context_dim, hidden_dim, bias=False)

        # --- 输出门 (Output Gate) - 公式 (9) ---
        self.W_o = nn.Linear(input_dim, hidden_dim, bias=True)
        self.U_o = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.V_oc = nn.Linear(context_dim, hidden_dim, bias=False)
        self.Z_ok = nn.Linear(context_dim, hidden_dim, bias=False)
        self.Q_ov = nn.Linear(context_dim, hidden_dim, bias=False)
        
        # --- 候选细胞状态 (Candidate Cell State) - 公式 (10) ---
        self.W_c = nn.Linear(input_dim, hidden_dim, bias=True)
        self.U_c = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.V_cc = nn.Linear(context_dim, hidden_dim, bias=False)
        self.Z_ck = nn.Linear(context_dim, hidden_dim, bias=False)
        self.Q_cv = nn.Linear(context_dim, hidden_dim, bias=False)

    def forward(self, x_prev, states, context_vectors):
        """
        :param x_prev: 上一时刻的输出 e^r_t, shape: [batch_size, input_dim]
        :param states: 上一时刻的状态 (h_prev, c_prev)
        :param context_vectors: 包含三种图上下文的元组 (e_c, e_k, e_v)
        """
        h_prev, c_prev = states
        e_c, e_k, e_v = context_vectors
        
        i_t = torch.sigmoid(self.W_i(x_prev) + self.U_i(h_prev) + \
                            self.V_ic(e_c) + self.Z_ik(e_k) + self.Q_iv(e_v))
        
        f_t = torch.sigmoid(self.W_f(x_prev) + self.U_f(h_prev) + \
                            self.V_fc(e_c) + self.Z_fk(e_k) + self.Q_fv(e_v))

        o_t = torch.sigmoid(self.W_o(x_prev) + self.U_o(h_prev) + \
                            self.V_oc(e_c) + self.Z_ok(e_k) + self.Q_ov(e_v))
        
        c_tilde_t = torch.tanh(self.W_c(x_prev) + self.U_c(h_prev) + \
                               self.V_cc(e_c) + self.Z_ck(e_k) + self.Q_cv(e_v))
        
        c_t = f_t * c_prev + i_t * c_tilde_t
        h_t = o_t * torch.tanh(c_t)
        
        return h_t, c_t

In [336]:
class NEXT (nn.Module):
    def __init__(self, num_videos, embed_dim, gnn_hidden_dim, lstm_hidden_dim):
        super(NEXT, self).__init__()
        
        self.video_embedding = nn.Embedding(num_videos, embed_dim)

        self.dgat_course = GATConv(embed_dim, gnn_hidden_dim, heads=2)
        self.dgat_concept = GATConv(embed_dim, gnn_hidden_dim, heads=2)
        self.gcn_video = GCNConv(embed_dim, gnn_hidden_dim*2)

        context_dim = gnn_hidden_dim*2

        self.LSTM = LSTM_layer(
            input_dim=lstm_hidden_dim,
            context_dim=context_dim,
            hidden_dim=lstm_hidden_dim
        )

        self.predictor = nn.Linear(lstm_hidden_dim, num_videos)

    def forward(self, h_prev, c_prev, static_edge_index_c, static_edge_index_k, dynamic_edge_index_v):
        # 1. Get initial node features from the embedding layer
        all_video_embeds = self.video_embedding.weight
        
        # 2. Learn representations from each graph
        # Static graphs
        e_c_nodes = torch.relu(self.dgat_course(all_video_embeds, static_edge_index_c))
        e_k_nodes = torch.relu(self.dgat_concept(all_video_embeds, static_edge_index_k))
        
        # Dynamic graph (handle empty case)
        if dynamic_edge_index_v.numel() > 0:
            e_v_nodes = torch.relu(self.gcn_video(all_video_embeds, dynamic_edge_index_v))
        else:
            e_v_nodes = torch.zeros_like(e_c_nodes) # Use zeros if no sequence info
        
        # 3. Aggregate node representations to get graph-level context vectors
        e_c_context = e_c_nodes.mean(dim=0, keepdim=True)
        e_k_context = e_k_nodes.mean(dim=0, keepdim=True)
        e_v_context = e_v_nodes.mean(dim=0, keepdim=True)
        
        # 4. Concatenate context vectors to form the LSTM input
        # context_tuple = torch.cat([e_c_context, e_k_context, e_v_context], dim=0).unsqueeze(0)
        context_tuple = (e_c_context, e_k_context, e_v_context)
        
        # 5. Update user's state with the LSTM
        h_t, c_t = self.LSTM(
            x_prev = h_prev, 
            states= (h_prev, c_prev),
            context_vectors = context_tuple)
        
        # 6. Predict the next video based on the updated user state
        video_logits = self.predictor(h_t)
        
        return video_logits, (h_t, c_t)


In [342]:
EMBED_DIM = 1024
GNN_HIDDEN_DIM = 1024
LSTM_HIDDEN_DIM = 1024
USER_ID_TO_PREDICT = 1 # 我们为用户1做预测
LEARING_RATE = 0.001
EPOCHS = 10
BATCH_SIZE = 1

model = NEXT(num_video, EMBED_DIM, GNN_HIDDEN_DIM, LSTM_HIDDEN_DIM)
lossf = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)